In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Import dataset loader from Hugging Face
from datasets import load_dataset

# Import train-test split function
from sklearn.model_selection import train_test_split

# Import feature scaling method
from sklearn.preprocessing import StandardScaler

# Import models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Import evaluation metrics
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    RocCurveDisplay
)

# Import library to save model
import joblib

In [ ]:
# Load dataset from Hugging Face
dataset = load_dataset("sanadf234/Heart-Disease-Prediction-dataset")

# Convert dataset into pandas dataframe
df = dataset['train'].to_pandas()

# Display first 5 rows
df.head()

In [ ]:
# Display number of rows and columns
print("Shape:", df.shape)

# Display dataset information
df.info()

# Display statistical summary
df.describe()

In [ ]:
# Check missing values in dataset
missing = df.isnull().sum()
print("Missing values per column:\n", missing)

In [ ]:
# Visualize target class distribution
plt.figure(figsize=(6, 4))
sns.countplot(x='heart_disease', data=df, palette=['#2ecc71', '#e74c3c'])
plt.title("Heart Disease Distribution")
plt.xlabel("Heart Disease (0 = No, 1 = Yes)")
plt.ylabel("Count")
plt.show()

In [ ]:
# Create correlation heatmap for numerical features
plt.figure(figsize=(12, 8))
sns.heatmap(df.corr(numeric_only=True), annot=True, fmt='.2f', cmap='coolwarm')
plt.title("Correlation Heatmap")
plt.show()

In [ ]:
# Separate input features and target column
X = df.drop("heart_disease", axis=1)
y = df["heart_disease"]

In [ ]:
# Convert categorical columns into numerical format using One-Hot Encoding
X = pd.get_dummies(X, drop_first=True)
X.head()

In [ ]:
# Split dataset into training and testing sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
# Create scaler object and scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# Initialize models
lr_model = LogisticRegression(random_state=42)
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)

# Train Logistic Regression
lr_model.fit(X_train_scaled, y_train)

# Train Random Forest Classifier
rf_model.fit(X_train, y_train)

print("Both Logistic Regression and Random Forest models trained successfully!")

In [ ]:
# Predict test dataset
lr_pred = lr_model.predict(X_test_scaled)
rf_pred = rf_model.predict(X_test)

In [ ]:
# Evaluate Logistic Regression
print("=== Logistic Regression Evaluation ===")
print("Accuracy:", accuracy_score(y_test, lr_pred))
print("ROC-AUC Score:", roc_auc_score(y_test, lr_pred))
print("\nClassification Report:\n", classification_report(y_test, lr_pred))

In [ ]:
# Display Confusion Matrix for Logistic Regression
cm_lr = confusion_matrix(y_test, lr_pred)
sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Blues')
plt.title("Confusion Matrix - Logistic Regression")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

In [ ]:
# Plot ROC Curve for Logistic Regression
RocCurveDisplay.from_estimator(lr_model, X_test_scaled, y_test)
plt.title("ROC Curve - Logistic Regression")
plt.show()

In [ ]:
# Random Forest Feature Importance
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['Feature'], feature_importance['Importance'], color='steelblue', edgecolor='black')
plt.title('Feature Importance - Random Forest', fontsize=14, fontweight='bold')
plt.xlabel('Importance Score')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print("Top 5 Most Important Features:\n", feature_importance.head())

In [ ]:
# Model Performance Comparison
models = ['Logistic Regression', 'Random Forest']
metrics = {
    'Accuracy': [accuracy_score(y_test, lr_pred), accuracy_score(y_test, rf_pred)],
    'Precision': [precision_score(y_test, lr_pred), precision_score(y_test, rf_pred)],
    'Recall': [recall_score(y_test, lr_pred), recall_score(y_test, rf_pred)],
    'F1 Score': [f1_score(y_test, lr_pred), f1_score(y_test, rf_pred)]
}

results_df = pd.DataFrame(metrics, index=models)
print("=== Model Comparison ===")
print(results_df.round(4))

# Plot performance comparison
fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(models))
width = 0.2

for i, (metric, values) in enumerate(metrics.items()):
    bars = ax.bar(x + i*width, values, width, label=metric, edgecolor='black')
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005, f'{val:.3f}', ha='center', va='bottom', fontsize=8)

ax.set_xlabel('Models')
ax.set_ylabel('Score')
ax.set_title('Model Performance Comparison', fontsize=14, fontweight='bold')
ax.set_xticks(x + width*1.5)
ax.set_xticklabels(models)
ax.legend()
ax.set_ylim(0, 1.1)
plt.tight_layout()
plt.show()

In [ ]:
# Save trained model and scaler
joblib.dump(lr_model, "models/logistic_regression_model.pkl")
joblib.dump(scaler, "models/scaler.pkl")
print("Models and scaler saved successfully!")

In [ ]:
# Sample input prediction
sample_data = X.iloc[0:1]
sample_data_scaled = scaler.transform(sample_data)
prediction = lr_model.predict(sample_data_scaled)
print("Sample Input Prediction (0 = No Heart Disease, 1 = Heart Disease):", prediction[0])

In [ ]:
# Conclusion & Project Summary
print("""
==================================================
PROJECT CONCLUSION: Heart Disease Prediction
==================================================
Dataset: Hugging Face sanadf234/Heart-Disease-Prediction-dataset (303 patients)
Models Trained:
  1. Logistic Regression (Linear baseline & interpretable probabilities)
  2. Random Forest Classifier (Non-linear ensemble learning)

Key Insights:
  - Feature scaling & One-Hot Encoding ensure robust multi-attribute classification.
  - Logistic Regression provides strong interpretability and aligns with production deployment requirements.
  - Model metrics and scaler assets are saved to disk in models/ for inference.
==================================================
""")